In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def extract_layer_number(head_name):
    """Extract layer number from head name like 'wmdp_head_1' -> 1"""
    original_layer = int(head_name.split('_')[-1])
    max_layer = 32
    reversed_layer = max_layer - original_layer
    return reversed_layer

def plot_probe_accuracy(csv_files, labels, colors=None, linestyles=None, title="Probe Accuracy"):
    """
    Plot probe accuracy vs layer for multiple models/conditions
    
    Args:
        csv_files: List of paths to CSV files containing accuracy data
        labels: List of labels for each CSV file
        colors: Optional list of colors for each line
        linestyles: Optional list of line styles for each line
        title: Title for the plot
    """
    plt.figure(figsize=(10, 6))
    
    if colors is None:
        colors = ['green', 'blue', 'red', 'orange', 'purple']
    if linestyles is None:
        linestyles = ['-', '--', '-.', ':', '-']
    
    for i, (csv_file, label) in enumerate(zip(csv_files, labels)):
        # Read the CSV file
        df = pd.read_csv(csv_file)
        
        # Extract layer numbers and sort by layer
        df['layer'] = df['head_name'].apply(extract_layer_number)
        df = df.sort_values('layer')
        
        # Plot the line
        plt.plot(df['layer'], df['accuracy'], 
                label=label, 
                color=colors[i % len(colors)],
                linestyle=linestyles[i % len(linestyles)],
                linewidth=2,
                marker='o' if '--' in linestyles[i % len(linestyles)] else None,
                markersize=3)
    
    # Add horizontal line for random chance (assuming binary classification, adjust as needed)
    plt.axhline(y=0.25, color='salmon', linestyle='-', alpha=0.7, label='Random chance')
    
    # Increase font sizes
    plt.xlabel('Layer', fontsize=16)  # Larger x-axis label
    plt.ylabel('Accuracy', fontsize=16)  # Larger y-axis label
    # plt.title(title, fontsize=18)  # Larger title
    plt.legend(fontsize=12)  # Larger legend text
    
    # Make tick labels larger
    plt.tick_params(axis='both', which='major', labelsize=14)
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return plt.gcf()

# Example usage with your data:
# Define your CSV files and corresponding labels
csv_files = [
    'prediction_results_elm/accuracy_summary.csv',
    'prediction_results_hindi_filler_baulab_elm_zephyr_7b_beta/accuracy_summary.csv',
    'prediction_results_zephyr_rmu/accuracy_summary.csv',
    'prediction_results_hindi_filler_zephyr_rmu/accuracy_summary.csv',
    'prediction_results_base_zephyr/accuracy_summary.csv',
]

labels = [
    'ELM unlearned, Original prompts', 
    'ELM unlearned, Hindi filler prompts',
    'RMU unlearned, Original prompts',
    'RMU unlearned, Hindi filler prompts',
    'Base model, Original prompts',
]

# Create the plot
fig = plot_probe_accuracy(csv_files, labels, title="Probe Accuracy for Unlearning Methods on Zephyr 7b")
fig.savefig('probe_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

# If you want to save the plot
# plt.savefig('probe_accuracy.png', dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

import matplotlib.font_manager as fm

# Adjusted font sizes for better readability
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['cmr10', 'Computer Modern Serif', 'DejaVu Serif', 'Times New Roman'],
    'mathtext.fontset': 'cm',  # This ensures math uses Computer Modern
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12
})

def plot_probe_accuracy_grouped(data_config, title="Probe Accuracy"):
    """
    Plot probe accuracy with grouped colors and line styles
    
    Args:
        data_config: List of dictionaries with keys: 'csv_file', 'label', 'color', 'linestyle'
        title: Title for the plot
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    
    for config in data_config:
        csv_file = config['csv_file']
        label = config['label']
        color = config['color']
        linestyle = config['linestyle']
        
        if not os.path.exists(csv_file):
            print(f"Warning: File {csv_file} not found, skipping...")
            continue
            
        # Read and process the CSV file
        df = pd.read_csv(csv_file)
        df['original_layer'] = df['head_name'].apply(lambda x: int(x.split('_')[-1]))
        max_layer = df['original_layer'].max()
        df['layer'] = max_layer - df['original_layer']
        df = df.sort_values('layer')
        
        # Plot the line
        ax.plot(df['layer'], df['accuracy'], 
                label=label, 
                color=color,
                linestyle=linestyle,
                linewidth=2.5,
                alpha=0.8)
    
    # Add horizontal line for random chance
    ax.axhline(y=0.25, color='#9467bd', linestyle='-', alpha=0.7, label='Random chance')
    
    # Styling
    ax.set_xlabel('Layer', fontsize=30)
    ax.set_ylabel('Accuracy', fontsize=30)
    # ax.set_title(title, fontsize=30)
    ax.legend(fontsize=25)
    ax.tick_params(axis='both', which='major', labelsize=25)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Configuration for your data
data_config = [
    # Base models
    {
        'csv_file': 'prediction_results_base_zephyr/accuracy_summary.csv',
        'label': 'Base model',
        'color': '#1f77b4',
        'linestyle': '-'
    },
    
    # ELM unlearned models (same color, different line styles)
    {
        'csv_file': 'prediction_results_elm/accuracy_summary.csv',
        'label': 'ELM',
        'color': '#d62728',
        'linestyle': '-'  # solid line
    },
    {
        'csv_file': 'prediction_results_hindi_filler_baulab_elm_zephyr_7b_beta/accuracy_summary.csv',
        'label': 'ELM, Hindi filler',
        'color': '#d62728',
        'linestyle': '--'  # dashed line
    },
    
    # RMU unlearned models (same color, different line styles)
    {
        'csv_file': 'prediction_results_zephyr_rmu/accuracy_summary.csv',
        'label': 'RMU',
        'color': '#2ca02c',
        'linestyle': '-'  # solid line
    },
    {
        'csv_file': 'prediction_results_hindi_filler_zephyr_rmu/accuracy_summary.csv',
        'label': 'RMU, Hindi filler',
        'color': '#2ca02c',
        'linestyle': '--'  # dashed line
    }
]

# Create the plot
fig = plot_probe_accuracy_grouped(data_config, title="Probe Accuracy for Unlearning Methods on Zephyr 7b")

# Save the figure
fig.savefig('probe_accuracy_grouped.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()